## This file is to take a look into the data gathered for the project
##### Daniel Jazowski

Here are the links used for downloading the datasets. 
FEVER: https://fever.ai/dataset/fever.html 
LIAR: https://www.kaggle.com/datasets/doanquanvietnamca/liar-dataset 
SciFact: https://www.kaggle.com/datasets/thedevastator/unlock-insight-into-scientific-claims-with-scifa?resource=download

In [1]:
import pandas as pd
import json

def load_fever(path):
    rows = []
    with open(path, "r") as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

fever_train = load_fever("FEVER/train.jsonl")
fever_dev = load_fever("FEVER/paper_dev.jsonl")

fever_df = pd.concat([fever_train, fever_dev], ignore_index=True)
fever_df = fever_df[['claim', 'label']]

label_map_fever = {
    "SUPPORTS": "supported",
    "REFUTES": "refuted",
    "NOT ENOUGH INFO": "not_enough_info"
}

fever_df['label'] = fever_df['label'].map(label_map_fever)
fever_df['source'] = "fever"

fever_df.head()

,claim,label,source
0,Nikolaj Coster-Waldau worked with the Fox Broa...,supported,fever
1,Roman Atwood is a content creator.,supported,fever
2,"History of art includes architecture, dance, s...",supported,fever
3,Adrienne Bailon is an accountant.,refuted,fever
4,System of a Down briefly disbanded in limbo.,not_enough_info,fever


In [2]:
import pandas as pd

liar_train = pd.read_csv("LIAR/train.tsv", sep='\t', header=None)
liar_valid = pd.read_csv("LIAR/valid.tsv", sep='\t', header=None)
liar_test  = pd.read_csv("LIAR/test.tsv",  sep='\t', header=None)

liar_df = pd.concat([liar_train, liar_valid, liar_test], ignore_index=True)
liar_df = liar_df[[2, 1]]
liar_df.columns = ['claim', 'label']
label_map_liar = {
    "pants-fire": "false",
    "false": "false",
    "barely-true": "mixed",
    "half-true": "mixed",
    "mostly-true": "true",
    "true": "true"
}

liar_df['label'] = liar_df['label'].map(label_map_liar)
liar_df['source'] = "liar"
liar_df.head()

,claim,label,source
0,Says the Annies List political group supports ...,false,liar
1,When did the decline of coal start? It started...,mixed,liar
2,"Hillary Clinton agrees with John McCain ""by vo...",true,liar
3,Health care reform legislation is likely to ma...,false,liar
4,The economic turnaround started at the end of ...,mixed,liar


In [3]:
scifact_train = pd.read_csv("SciFact/claims_train.csv")
scifact_valid = pd.read_csv("SciFact/claims_validation.csv")
scifact_test  = pd.read_csv("SciFact/claims_test.csv")

scifact_df = pd.concat([scifact_train, scifact_valid, scifact_test], ignore_index=True)
scifact_df = scifact_df[['claim', 'evidence_label']]
scifact_df = scifact_df.rename(columns={'evidence_label': 'label'})
scifact_df = scifact_df.dropna(subset=['label'])
scifact_df['label'] = scifact_df['label'].str.lower()
scifact_df['source'] = "scifact"
label_map_scifact = {
    "support": "supported",
    "contradict": "refuted"
}

scifact_df['label'] = scifact_df['label'].map(label_map_scifact)
scifact_df.head()

,claim,label,source
1,1 in 5 million in UK have abnormal PrP positiv...,refuted,scifact
4,32% of liver transplantation programs required...,supported,scifact
7,40mg/day dosage of folic acid and 2mg/day dosa...,supported,scifact
8,40mg/day dosage of folic acid and 2mg/day dosa...,supported,scifact
16,76-85% of people with severe mental disorder r...,supported,scifact


In [4]:
combined = pd.concat([fever_df, liar_df, scifact_df], ignore_index=True)

combined['claim'] = combined['claim'].astype(str).str.strip()
combined = combined.drop_duplicates(subset=['claim'])
combined = combined.dropna(subset=['claim', 'label'])
collapse_map = {
    "supported": "supported",
    "true": "supported",

    "refuted": "refuted",
    "false": "refuted",

    "mixed": "not_enough_info",
    "not_enough_info": "not_enough_info"
}

combined['label'] = combined['label'].map(collapse_map)
combined.head()

,claim,label,source
0,Nikolaj Coster-Waldau worked with the Fox Broa...,supported,fever
1,Roman Atwood is a content creator.,supported,fever
2,"History of art includes architecture, dance, s...",supported,fever
3,Adrienne Bailon is an accountant.,refuted,fever
4,System of a Down briefly disbanded in limbo.,not_enough_info,fever


In [5]:
combined['source'].value_counts()

source
fever      145287
liar        12765
scifact       692
Name: count, dtype: int64

In [6]:
combined.groupby('source')['label'].value_counts()

source   label          
fever    supported          77054
         not_enough_info    36814
         refuted            31419
liar     not_enough_info     4723
         supported           4502
         refuted             3540
scifact  supported            455
         refuted              237
Name: count, dtype: int64

In [7]:
combined.to_csv("combinedClaimDataset.csv", index=False)